# 01_loss

- Implements and verifies all loss functions used in DeclipNet training
- L1 loss on raw waveform (baseline)
- Amplitude-weighted L1 loss (weight_power parameter: 0=uniform, >0=emphasis on clipped regions)
- Multi-resolution STFT loss (weight_power parameter: 0=uniform, >0=high freq emphasis)
- DWT loss via pytorch_wavelets (weight_power parameter: 0=uniform, >0=high freq emphasis)
- SI-SDR metric (not a training loss)
- Gradient flow verified for all differentiable loss functions

In [1]:
# Imports

import sys
import torch

sys.path.insert(0, "..")
from config import *
from util import l1_loss, weighted_l1_loss, multires_stft_loss, dwt_loss, si_sdr

In [2]:
# Shared test tensors
torch.manual_seed(0)
BATCH = 4

clean = torch.randn(BATCH, 1, BS)
peak = clean.abs().amax(dim=2, keepdim=True)
alpha = 0.5
clipped = clean.clamp(-alpha * peak, alpha * peak)
output = clean + 0.01 * torch.randn_like(clean)  # near-perfect reconstruction

In [3]:
# L1 Loss
mean_abs = clean.abs().mean().item()

l1_00 = l1_loss(clean, clean).item()
l1_75 = l1_loss(clean * 0.75, clean).item()
l1_50 = l1_loss(clean * 0.50, clean).item()
l1_25 = l1_loss(clean * 0.25, clean).item()

print(f"L1 (scale=1.00): {l1_00:.6f}  expected: 0.000000")
print(f"L1 (scale=0.75): {l1_75:.6f}  expected: {0.25 * mean_abs:.6f}")
print(f"L1 (scale=0.50): {l1_50:.6f}  expected: {0.50 * mean_abs:.6f}")
print(f"L1 (scale=0.25): {l1_25:.6f}  expected: {0.75 * mean_abs:.6f}")

assert l1_00 == 0.0
assert l1_75 < l1_50 < l1_25
assert abs(l1_50 / l1_25 - (0.50 / 0.75)) < 1e-5, "Loss ratios should be proportional"
print("L1 checks passed")

L1 (scale=1.00): 0.000000  expected: 0.000000
L1 (scale=0.75): 0.196804  expected: 0.196804
L1 (scale=0.50): 0.393609  expected: 0.393609
L1 (scale=0.25): 0.590413  expected: 0.590413
L1 checks passed


In [4]:
# Weighted L1 Loss

# Construct a signal where we know exactly where the clipping boundary is
test_clean = torch.zeros(1, 1, BS)
test_clean[0, 0, :BS//2] = 0.1   # low amplitude region
test_clean[0, 0, BS//2:] = 1.0   # high amplitude region

alpha_test = 0.5
peak_test = test_clean.abs().amax(dim=2, keepdim=True)
test_clipped = test_clean.clamp(-alpha_test * peak_test, alpha_test * peak_test)

# Introduce equal magnitude error in low vs high amplitude region
error_mag = 0.05
test_output_low_err  = test_clean.clone()
test_output_high_err = test_clean.clone()
test_output_low_err[0, 0, :BS//2]  += error_mag   # error in low amplitude region
test_output_high_err[0, 0, BS//2:] += error_mag   # error in high amplitude region

loss_low  = weighted_l1_loss(test_output_low_err,  test_clean, test_clipped, weight_power=1.0).item()
loss_high = weighted_l1_loss(test_output_high_err, test_clean, test_clipped, weight_power=1.0).item()

print(f"Weighted L1 error in low amplitude region:  {loss_low:.6f}")
print(f"Weighted L1 error in high amplitude region: {loss_high:.6f}")
assert loss_high > loss_low, "Error near clipping boundary should be penalized more heavily"
print("Amplitude weighting check passed")

# Verify weight_power=0 gives uniform weighting (matches standard L1)
loss_uniform = weighted_l1_loss(output, clean, clipped, weight_power=0.0).item()
loss_standard = l1_loss(output, clean).item()
assert abs(loss_uniform - loss_standard) < 1e-5, "weight_power=0 should match standard L1"
print(f"Uniform weighting matches L1: {loss_uniform:.6f} == {loss_standard:.6f}")

# Verify higher power increases relative emphasis on high vs low amplitude errors
test_output_both_err = test_clean.clone()
test_output_both_err[0, 0, :BS//2] += error_mag   # error in low amplitude region
test_output_both_err[0, 0, BS//2:] += error_mag   # equal error in high amplitude region

loss_p0 = weighted_l1_loss(test_output_both_err, test_clean, test_clipped, weight_power=0.0).item()
loss_p1 = weighted_l1_loss(test_output_both_err, test_clean, test_clipped, weight_power=1.0).item()
loss_p2 = weighted_l1_loss(test_output_both_err, test_clean, test_clipped, weight_power=2.0).item()
loss_p4 = weighted_l1_loss(test_output_both_err, test_clean, test_clipped, weight_power=4.0).item()

print(f"Both regions error, p=0 (uniform): {loss_p0:.6f}")
print(f"Both regions error, p=1:           {loss_p1:.6f}")
print(f"Both regions error, p=2:           {loss_p2:.6f}")
print(f"Both regions error, p=4:           {loss_p4:.6f}")

# Higher power down-weights low amplitude region more, 
# shifting total loss toward high amplitude contribution
assert loss_p1 < loss_p0, "p=1 should down-weight low amplitude vs uniform"
assert loss_p2 < loss_p1, "p=2 should down-weight low amplitude more than p=1"
assert loss_p4 < loss_p2, "p=4 should down-weight low amplitude more than p=2"
print("Power scaling checks passed ")

Weighted L1 error in low amplitude region:  0.005000
Weighted L1 error in high amplitude region: 0.025000
Amplitude weighting check passed
Uniform weighting matches L1: 0.008072 == 0.008072
Both regions error, p=0 (uniform): 0.050000
Both regions error, p=1:           0.030000
Both regions error, p=2:           0.026000
Both regions error, p=4:           0.025040
Power scaling checks passed 


In [5]:
# Multi-resolution STFT Loss — basic scaling checks
stft_00 = multires_stft_loss(clean, clean, weight_power=0.0).item()
stft_75 = multires_stft_loss(clean * 0.75, clean, weight_power=0.0).item()
stft_50 = multires_stft_loss(clean * 0.50, clean, weight_power=0.0).item()
stft_25 = multires_stft_loss(clean * 0.25, clean, weight_power=0.0).item()

print(f"STFT (scale=1.00): {stft_00:.6f}  expected: ~0.0")
print(f"STFT (scale=0.75): {stft_75:.6f}")
print(f"STFT (scale=0.50): {stft_50:.6f}")
print(f"STFT (scale=0.25): {stft_25:.6f}")

assert stft_00 < 1e-6, "STFT loss should be near zero for identical inputs"
assert stft_75 < stft_50 < stft_25, "STFT loss should increase as scaling deviates further from 1"
print("STFT scaling checks passed ")

# Multi-resolution STFT Loss — low pass filter checks
# Low pass filtering removes high frequency content — STFT loss should
# increase with more aggressive filtering, and weight_power>0 should
# amplify this since high freq errors are weighted more heavily
import torch.nn.functional as F

def low_pass(x, kernel_size):
    # Simple box filter low pass via avg_pool1d
    pad = kernel_size // 2
    return F.avg_pool1d(
        F.pad(x, (pad, pad), mode='reflect'),
        kernel_size=kernel_size,
        stride=1,
        padding=0
    )

lp_mild     = low_pass(clean, kernel_size=3)
lp_moderate = low_pass(clean, kernel_size=11)
lp_severe   = low_pass(clean, kernel_size=31)

for label, lp in [("mild (k=3)", lp_mild), ("moderate (k=11)", lp_moderate), ("severe (k=31)", lp_severe)]:
    loss_uniform  = multires_stft_loss(lp, clean, weight_power=0.0).item()
    loss_weighted = multires_stft_loss(lp, clean, weight_power=1.0).item()
    print(f"LP {label:20s} | uniform: {loss_uniform:.6f}  weighted: {loss_weighted:.6f}")

# More aggressive LP should produce higher loss
lp_losses = [multires_stft_loss(lp, clean, weight_power=0.0).item() 
             for lp in [lp_mild, lp_moderate, lp_severe]]
assert lp_losses[0] < lp_losses[1] < lp_losses[2], \
    "More aggressive low pass should produce higher STFT loss"

# High freq weighting should amplify LP loss since LP removes high freq content
for lp, label in [(lp_mild, "mild"), (lp_moderate, "moderate"), (lp_severe, "severe")]:
    loss_uniform  = multires_stft_loss(lp, clean, weight_power=0.0).item()
    loss_weighted = multires_stft_loss(lp, clean, weight_power=1.0).item()
    assert loss_weighted > loss_uniform, \
        f"High freq weighting should increase loss for LP filtered input ({label})"

print("STFT low pass checks passed ")

STFT (scale=1.00): 0.000000  expected: ~0.0
STFT (scale=0.75): 2.103751
STFT (scale=0.50): 4.207502
STFT (scale=0.25): 6.311253
STFT scaling checks passed 
LP mild (k=3)           | uniform: 4.360058  weighted: 5.716248
LP moderate (k=11)      | uniform: 6.896060  weighted: 7.692135
LP severe (k=31)        | uniform: 7.725205  weighted: 8.152377
STFT low pass checks passed 


In [6]:
# DWT Loss — basic scaling checks
dwt_00 = dwt_loss(clean, clean, weight_power=0.0).item()
dwt_75 = dwt_loss(clean * 0.75, clean, weight_power=0.0).item()
dwt_50 = dwt_loss(clean * 0.50, clean, weight_power=0.0).item()
dwt_25 = dwt_loss(clean * 0.25, clean, weight_power=0.0).item()

print(f"DWT (scale=1.00): {dwt_00:.6f}  expected: ~0.0")
print(f"DWT (scale=0.75): {dwt_75:.6f}")
print(f"DWT (scale=0.50): {dwt_50:.6f}")
print(f"DWT (scale=0.25): {dwt_25:.6f}")

assert dwt_00 < 1e-5, "DWT loss should be near zero for identical inputs"
assert dwt_75 < dwt_50 < dwt_25, "DWT loss should increase as scaling deviates further from 1"
print("DWT scaling checks passed ")

# DWT Loss — low pass filter checks
# LP filtering removes high freq content — weighted loss should be
# higher than uniform since high freq error is emphasized
for label, lp in [("mild (k=3)", lp_mild), ("moderate (k=11)", lp_moderate), ("severe (k=31)", lp_severe)]:
    loss_uniform  = dwt_loss(lp, clean, weight_power=0.0).item()
    loss_weighted = dwt_loss(lp, clean, weight_power=1.0).item()
    print(f"DWT LP {label:20s} | uniform: {loss_uniform:.6f}  weighted: {loss_weighted:.6f}")
    assert loss_weighted > loss_uniform, \
        f"High freq weighting should increase DWT loss for LP filtered input ({label})"

print("DWT low pass checks passed ")

# DWT Loss — weight_power scaling
dwt_uniform   = dwt_loss(output, clean, weight_power=0.0).item()
dwt_weighted1 = dwt_loss(output, clean, weight_power=1.0).item()
dwt_weighted2 = dwt_loss(output, clean, weight_power=2.0).item()

print(f"DWT uniform:    {dwt_uniform:.6f}")
print(f"DWT weighted=1: {dwt_weighted1:.6f}")
print(f"DWT weighted=2: {dwt_weighted2:.6f}")

assert dwt_uniform > 0, "DWT loss should be positive for imperfect output"
# Weight power effect already verified via LP filter checks above
print("DWT weight_power checks passed ")

# Gradient flow check (use non-identical output; L1 grad is zero when output == target)
output_grad = output.clone().requires_grad_(True)
loss = dwt_loss(output_grad, clean)
loss.backward()
assert output_grad.grad is not None, "DWT loss has no gradient"
assert output_grad.grad.abs().sum().item() > 0, "DWT loss gradient is zero"
print("DWT gradient flow ")

DWT (scale=1.00): 0.000000  expected: ~0.0
DWT (scale=0.75): 0.198142
DWT (scale=0.50): 0.396284
DWT (scale=0.25): 0.594426
DWT scaling checks passed 
DWT LP mild (k=3)           | uniform: 0.291923  weighted: 0.429775
DWT LP moderate (k=11)      | uniform: 0.577598  weighted: 0.700908
DWT LP severe (k=31)        | uniform: 0.735404  weighted: 0.782252
DWT low pass checks passed 
DWT uniform:    0.007999
DWT weighted=1: 0.007969
DWT weighted=2: 0.007998
DWT weight_power checks passed 
DWT gradient flow 


In [7]:
# SI-SDR verification
sdr_perfect_vec = si_sdr(clean, clean)
sdr_clipped_vec = si_sdr(clean, clipped)
sdr_scaled_vec  = si_sdr(clean, clean * 0.75)
sdr_noisy_vec   = si_sdr(clean, clean + 0.1 * torch.randn_like(clean))

assert sdr_perfect_vec.shape == (BATCH,), f"Expected shape ({BATCH},), got {sdr_perfect_vec.shape}"
print(f"SI-SDR returns per-example tensor: shape {sdr_perfect_vec.shape}")

sdr_perfect = sdr_perfect_vec.mean().item()
sdr_clipped = sdr_clipped_vec.mean().item()
sdr_scaled  = sdr_scaled_vec.mean().item()
sdr_noisy   = sdr_noisy_vec.mean().item()

print(f"SI-SDR (perfect match):     {sdr_perfect:.2f} dB")
print(f"SI-SDR (clipped):           {sdr_clipped:.2f} dB")
print(f"SI-SDR (scaled 0.75):       {sdr_scaled:.2f} dB")
print(f"SI-SDR (clean + noise 0.1): {sdr_noisy:.2f} dB")

assert sdr_perfect > 60, f"Perfect match SI-SDR should be >60 dB, got {sdr_perfect:.2f}"
assert sdr_scaled > 60, f"Scale-invariant: scaled input should also be >60 dB, got {sdr_scaled:.2f}"
assert sdr_clipped < sdr_perfect, "Clipped SI-SDR should be lower than perfect"
assert sdr_noisy < sdr_perfect, "Noisy SI-SDR should be lower than perfect"
assert sdr_clipped < sdr_noisy, "Heavy clipping (alpha=0.5) should be worse than mild noise (std=0.1)"
print("SI-SDR checks passed")

SI-SDR returns per-example tensor: shape torch.Size([4])
SI-SDR (perfect match):     110.03 dB
SI-SDR (clipped):           17.96 dB
SI-SDR (scaled 0.75):       107.54 dB
SI-SDR (clean + noise 0.1): 19.90 dB
SI-SDR checks passed


In [8]:
# Gradient flow verification for all differentiable loss functions
loss_fns = [
    ("l1_loss",           lambda x: l1_loss(x, clean)),
    ("weighted_l1_loss",  lambda x: weighted_l1_loss(x, clean, clipped)),
    ("multires_stft_loss", lambda x: multires_stft_loss(x, clean)),
    ("dwt_loss",          lambda x: dwt_loss(x, clean)),
]

for name, fn in loss_fns:
    output_grad = output.clone().requires_grad_(True)
    loss = fn(output_grad)
    loss.backward()
    assert output_grad.grad is not None, f"{name} produced no gradient"
    grad_sum = output_grad.grad.abs().sum().item()
    assert grad_sum > 0, f"{name} gradient is zero"
    print(f"{name}: gradient flows (sum abs grad = {grad_sum:.6f})")
    output_grad.grad = None

print("All gradient flow checks passed")

l1_loss: gradient flows (sum abs grad = 1.000000)
weighted_l1_loss: gradient flows (sum abs grad = 0.400659)
multires_stft_loss: gradient flows (sum abs grad = 6.062222)
dwt_loss: gradient flows (sum abs grad = 0.884102)
All gradient flow checks passed
